In [3]:
#Task 3
import random

graph = {
    'Warehouse': {'A': 2, 'B': 5, 'C': 4},
    'A':         {'B': 3, 'D': 6},
    'B':         {'C': 2, 'E': 4},
    'C':         {'F': 3},
    'D':         {'G': 5},
    'E':         {'G': 2, 'H': 6},
    'F':         {'H': 3},
    'G':         {},
    'H':         {}
}

delivery_points = {
    'A': {'deadline': 5,  'distance': 2},
    'B': {'deadline': 8,  'distance': 5},
    'C': {'deadline': 6,  'distance': 4},
    'D': {'deadline': 15, 'distance': 9},
    'E': {'deadline': 10, 'distance': 7},
    'F': {'deadline': 7,  'distance': 6},
    'G': {'deadline': 20, 'distance': 13},
    'H': {'deadline': 12, 'distance': 11}
}

def heuristic(node, delivery_points):
    if node in delivery_points:
        deadline = delivery_points[node]['deadline']
        distance = delivery_points[node]['distance']
        return deadline - distance
    return float('inf')

def greedy_bfs(graph, start, delivery_points):
    frontier = [(start, heuristic(start, delivery_points))]
    visited = set()
    came_from = {start: None}
    delivery_order = []
    current_time = 0

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, _ = frontier.pop(0)

        if current_node in visited:
            continue

        visited.add(current_node)

        if current_node in delivery_points:
            current_time += delivery_points[current_node]['distance']
            status = "ON TIME" if current_time <= delivery_points[current_node]['deadline'] else "LATE"
            delivery_order.append((current_node, current_time, status))
            print(f"Delivered to {current_node} at time {current_time} -- {status} (Deadline: {delivery_points[current_node]['deadline']})")

        for neighbor in graph[current_node]:
            if neighbor not in visited:
                came_from[neighbor] = current_node
                frontier.append((neighbor, heuristic(neighbor, delivery_points)))

    print(f"\nDelivery Order Summary:")
    for stop, time, status in delivery_order:
        print(f"  {stop} --> Time: {time} | Status: {status}")

    on_time = sum(1 for _, _, s in delivery_order if s == "ON TIME")
    late    = sum(1 for _, _, s in delivery_order if s == "LATE")
    print(f"\nTotal Deliveries: {len(delivery_order)} | On Time: {on_time} | Late: {late}")


print("\nFollowing is the Greedy Best-First Search Delivery Route:\n")
greedy_bfs(graph, 'Warehouse', delivery_points)


Following is the Greedy Best-First Search Delivery Route:

Delivered to C at time 4 -- ON TIME (Deadline: 6)
Delivered to F at time 10 -- LATE (Deadline: 7)
Delivered to H at time 21 -- LATE (Deadline: 12)
Delivered to A at time 23 -- LATE (Deadline: 5)
Delivered to B at time 28 -- LATE (Deadline: 8)
Delivered to E at time 35 -- LATE (Deadline: 10)
Delivered to D at time 44 -- LATE (Deadline: 15)
Delivered to G at time 57 -- LATE (Deadline: 20)

Delivery Order Summary:
  C --> Time: 4 | Status: ON TIME
  F --> Time: 10 | Status: LATE
  H --> Time: 21 | Status: LATE
  A --> Time: 23 | Status: LATE
  B --> Time: 28 | Status: LATE
  E --> Time: 35 | Status: LATE
  D --> Time: 44 | Status: LATE
  G --> Time: 57 | Status: LATE

Total Deliveries: 8 | On Time: 1 | Late: 7


In [24]:
#Task 2
from queue import PriorityQueue
import random
import time

graph = {
    'A': {'B': 4, 'C': 3},
    'B': {'E': 12, 'F': 5},
    'C': {'D': 7, 'E': 10},
    'D': {'E': 2},
    'E': {'G': 5},
    'F': {'G': 16},
    'G': {}
}

heuristic = {'A': 14, 'B': 12, 'C': 11, 'D': 6, 'E': 4, 'F': 11, 'G': 0}

def update_edge_costs(graph):
    for node in graph:
        for neighbor in graph[node]:
            change = random.randint(-2, 2)
            graph[node][neighbor] = max(1, graph[node][neighbor] + change)
    return graph

def a_star(graph, start, goal):
    frontier = [(start, 0 + heuristic[start])]
    visited = set()
    g_costs = {start: 0}
    came_from = {start: None}

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, current_f = frontier.pop(0)

        if current_node in visited:
            continue

        print(current_node, end=" ")
        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print(f"\nGoal found with A*. Path: {path}")
            return path

        for neighbor, cost in graph[current_node].items():
            new_g_cost = g_costs[current_node] + cost
            f_cost = new_g_cost + heuristic[neighbor]

            if neighbor not in g_costs or new_g_cost < g_costs[neighbor]:
                g_costs[neighbor] = new_g_cost
                came_from[neighbor] = current_node
                frontier.append((neighbor, f_cost))

    print("\nGoal not found")
    return None

def dynamic_a_star(graph, start, goal, intervals=3):
    print("\nFollowing is the Dynamic A* Search:\n")
    current_start = start
    full_path = []

    for i in range(intervals):
        print(f"\nInterval {i + 1} - Current edge costs: {graph}")
        segment_path = a_star(graph, current_start, goal)

        if segment_path is None:
            print("No path found, stopping.")
            return None

        if full_path:
            full_path += segment_path[1:]
        else:
            full_path += segment_path

        if segment_path[-1] == goal:
            print(f"\nReached goal. Full path: {full_path}")
            return full_path

        current_start = segment_path[-1]

        print(f"\nUpdating edge costs dynamically...")
        graph = update_edge_costs(graph)
        time.sleep(1)

    print(f"\nFinal path after {intervals} intervals: {full_path}")
    return full_path


dynamic_a_star(graph, 'A', 'G', intervals=3)


Following is the Dynamic A* Search:


Interval 1 - Current edge costs: {'A': {'B': 4, 'C': 3}, 'B': {'E': 12, 'F': 5}, 'C': {'D': 7, 'E': 10}, 'D': {'E': 2}, 'E': {'G': 5}, 'F': {'G': 16}, 'G': {}}
A C B D E G 
Goal found with A*. Path: ['A', 'C', 'D', 'E', 'G']

Reached goal. Full path: ['A', 'C', 'D', 'E', 'G']


['A', 'C', 'D', 'E', 'G']

In [ ]:
#Task 1
from queue import PriorityQueue

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0
        self.h = 0
        self.f = 0

    def __lt__(self, other):
        return self.f < other.f

def heuristic(current_pos, goal_pos):
    return abs(current_pos[0] - goal_pos[0]) + abs(current_pos[1] - goal_pos[1])

def nearest_goal(current_pos, remaining_goals):
    return min(remaining_goals, key=lambda goal: heuristic(current_pos, goal))

def best_first_search(maze, start, goals):
    rows, cols = len(maze), len(maze[0])
    remaining_goals = list(goals)
    full_path = []
    current_start = start

    while remaining_goals:
        target = nearest_goal(current_start, remaining_goals)
        print(f"\nSearching from {current_start} --> {target}")

        start_node = Node(current_start)
        frontier = PriorityQueue()
        frontier.put(start_node)
        visited = set()

        segment_path = None

        while not frontier.empty():
            current_node = frontier.get()
            current_pos = current_node.position

            if current_pos == target:
                path = []
                while current_node:
                    path.append(current_node.position)
                    current_node = current_node.parent
                segment_path = path[::-1]
                break

            visited.add(current_pos)

            for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
                new_pos = (current_pos[0] + dx, current_pos[1] + dy)
                if 0 <= new_pos[0] < rows and 0 <= new_pos[1] < cols and maze[new_pos[0]][new_pos[1]] == 0 and new_pos not in visited:
                    new_node = Node(new_pos, current_node)
                    new_node.g = current_node.g + 1
                    new_node.h = heuristic(new_pos, target)
                    new_node.f = new_node.h
                    frontier.put(new_node)
                    visited.add(new_pos)

        if segment_path is None:
            print(f"No path found to goal {target}")
            return None

        if full_path:
            full_path += segment_path[1:]
        else:
            full_path += segment_path

        remaining_goals.remove(target)
        current_start = target

    return full_path


maze = [
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 1],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0]
]

start = (0, 0)
goals = [(4, 4), (0, 3), (3, 0)]

path = best_first_search(maze, start, goals)

if path:
    print(f"\nFull path covering all goals: {path}")
else:
    print("No complete path found")


Searching from (0, 0) --> (0, 3)

Searching from (0, 3) --> (4, 4)

Searching from (4, 4) --> (3, 0)

Full path covering all goals: [(0, 0), (0, 1), (1, 1), (1, 2), (1, 3), (0, 3), (1, 3), (2, 3), (3, 3), (3, 4), (4, 4), (3, 4), (3, 3), (2, 3), (1, 3), (1, 2), (1, 1), (2, 1), (3, 1), (3, 0)]


In [13]:
#Heuristic Greedy Best First Search
# Graph with different edge costs
graph = {
    'A': {'B': 2, 'C': 1},
    'B': {'D': 4, 'E': 3},
    'C': {'F': 1, 'G': 5},
    'D': {'H': 2},
    'E': {},
    'F': {'I': 6},
    'G': {},
    'H': {},
    'I': {}
}

# Heuristic function (estimated cost to reach goal 'I')
heuristic = {
    'A': 7,
    'B': 6,
    'C': 5,
    'D': 4,
    'E': 7,
    'F': 3,
    'G': 6,
    'H': 2,
    'I': 0  # Goal node
}

# Greedy Best-First Search Function (without heapq)
def greedy_bfs(graph, start, goal):
    frontier = [(start, heuristic[start])]  # List-based priority queue (sorted manually)
    visited = set()  # Set to keep track of visited nodes
    came_from = {start: None}  # Path reconstruction

    while frontier:
        # Sort frontier manually by heuristic value (ascending order)
        frontier.sort(key=lambda x: x[1])
        current_node, _ = frontier.pop(0)  # Get node with best heuristic

        if current_node in visited:
            continue

        print(current_node, end=" ")  # Print visited node
        visited.add(current_node)

        # If goal is reached, reconstruct path
        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print(f"\nGoal found with GBFS. Path: {path}")
            return

        # Expand neighbors based on heuristic
        for neighbor in graph[current_node]:
            if neighbor not in visited:
                came_from[neighbor] = current_node
                frontier.append((neighbor, heuristic[neighbor]))

    print("\nGoal not found")

# Run Greedy Best-First Search
print("\nFollowing is the Greedy Best-First Search (GBFS):")
greedy_bfs(graph, 'A', 'I')




Following is the Greedy Best-First Search (GBFS):
A C F I 
Goal found with GBFS. Path: ['A', 'C', 'F', 'I']


In [12]:
#Best First Search
from queue import PriorityQueue

graph = {
    'A': [('B', 5), ('C', 8)],
    'B': [('D', 10)],
    'C': [('E', 3)],
    'D': [('F', 7)],
    'E': [('F', 2)],
    'F': []
}


def best_first_search(graph, start, goal):
    visited = set()
    pq = PriorityQueue()
    pq.put((0, start))

    while not pq.empty():
        cost, node = pq.get()

        if node not in visited:
            print(node, end = ' ')
            visited.add(node)

            if node == goal:
                print("\nGoal Reached \n")
                return True

            for neighbour, weight in graph[node]:
                if neighbour not in visited:
                    pq.put((weight, neighbour))


    print("\nGoal not reachable!")
    return False

print("Best-First Search Path:")
best_first_search(graph, 'A', 'F')

Best-First Search Path:
A B C E F 
Goal Reached 



True